# Task 4: Deployment Strategies
A) We'll create a model inference pipeline that will accept a segment of data and generate a prediction.

B) We'll discuss deployment strategies for a production-ready solution, including challenges and best practices.

C) We'll discuss handling model versioning, scalability, and monitoring in the deployed system, including risks and mitigation strategies.

In [1]:
%load_ext autoreload
%autoreload 2

### Required package imports

In [ ]:
import sys
import os
from fastapi.testclient import TestClient

import random
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report

from resnet_hearthealth_monitor.config import DEVICE
from resnet_hearthealth_monitor.modeling.inference import predict_segment
from resnet_hearthealth_monitor.inference_api import app
from resnet_hearthealth_monitor.modeling.train import load_best_study

2025-07-29 16:54:55.372 | INFO     | lightweight_resnet_heart_monitor.config:<module>:12 - PROJ_ROOT path is: /home/anthony/home/github/lightweight_resnet_heart_monitor
/home/anthony/.virtualenvs/lightweight_resnet_heart_monitor/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-07-29 16:54:55.438 | INFO     | lightweight_resnet_heart_monitor.config:<module>:41 - DEVICE is set to: cuda


In [3]:
device = DEVICE
if device == 'cpu':
    print("WARNING: CUDA not available. Using CPU for training, which may be very slow.")

# A) Creating a model inference pipeline

### First, sanity check the results of our inference predict_segment() function

We do this by using 150 samples from the test set, feeding these into our inference engine. We should see comparable results to chapter_2_model_training's "Generate and evaluate test set predictions" section.

In [4]:
# Sample an equal amount of each class to test the inference process on:
test_set = pd.read_csv('../data/processed/mitbih_test_cropped.csv', header=None)  
unique_labels = test_set.iloc[:, -1].unique()
num_classes = len(unique_labels)
samples_per_class = 30

sampled_indices = []
for label in unique_labels:
    # Get indices for the current class
    indices = test_set[test_set.iloc[:, -1] == label].index.tolist()
    # Randomly sample equal number of indices from this class
    sampled = random.sample(indices, samples_per_class)
    sampled_indices.extend(sampled)
sampled_indices = sorted(sampled_indices)

predicted_classes=[]
true_classes=[]

for i in sampled_indices:
    segment = test_set.iloc[i, :-1].values  # Ensure this is a 1D array of 125 features

    # Predict the class for the segment
    pred = predict_segment(segment, device)
    predicted_classes.append(pred)
    true_label = test_set.iloc[i, -1]
    true_classes.append(true_label)

# Create a DataFrame to display the results
results_df = pd.DataFrame({
    "Index": sampled_indices,
    "Predicted": predicted_classes,
    "True": true_classes
})
print(results_df)

# Optionally, generate a classification report if there is more than one class in the sample.
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes))

['../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_0_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_1_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_2_best.pth']
2025-07-29 16:55:03.436 | INFO     | lightweight_resnet_heart_monitor.modeling.inference:predict_segment:56 - Performing inference with best model...
['../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_0_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_1_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_2_best.pth']
2025-07-29 16:55:03.704 | INFO     | lightweight_resnet_heart_monitor.modeling.inference:predict_segment:56 - Performing inference with best model...
['../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/

### Sanity check passed: performances with inference code similar to full test set performances

### FastAPI-based model inference

Sanity check our inference_api.py and modeling/inference.py scripts by sending one example from each class through our FastAPI

In [5]:
samples_per_class = 1

sampled_indices = []
for label in unique_labels:
    # Get indices for the current class
    indices = test_set[test_set.iloc[:, -1] == label].index.tolist()
    # Randomly sample equal number of indices from this class
    sampled = random.sample(indices, samples_per_class)
    sampled_indices.extend(sampled)
sampled_indices = sorted(sampled_indices)

# 'app' is your FastAPI instance from inference_api.py
client = TestClient(app)

for i in sampled_indices:
    # Create a payload with 125 features
    segment = test_set.iloc[i, :-1].values.tolist()
    payload = {"segment": segment}

    # Send a POST request to the /predict endpoint
    response = client.post("/predict", json=payload)

    print("Status Code:", response.status_code)
    print("Response JSON:", response.json())

['../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_0_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_1_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_2_best.pth']
2025-07-29 16:55:18.347 | INFO     | lightweight_resnet_heart_monitor.modeling.inference:predict_segment:56 - Performing inference with best model...
Status Code: 200
Response JSON: {'predicted_class': 0}
['../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_0_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_1_best.pth', '../models/20250302_140242_ResNet125_firstpass_5trials_10epochs_3folds/trial_1/model_fold_2_best.pth']
2025-07-29 16:55:18.389 | INFO     | lightweight_resnet_heart_monitor.modeling.inference:predict_segment:56 - Performing inference with best model...
Status Code: 200
Response

### Sanity check passed: FastAPI working well

# B) Deployment strategy

Next steps in the deployment of this package would include containerization, deployment, and setting up a CI/CD pipeline. 

For containerization, we would use Docker to create a container encapsulating all details of the runtime environment and package, ensuring consistency across development, testing, and production. Small changes in details like the change of a Python package version number can potentially change the behaviour of some downstream functions and containerization protects against this by ensuring an exact replica of the runtime environment when we wish to move the package to a deployment platform.

With our package in a container, we would now deploy it on cloud platform so our model inference API can be ready to accept a new inference request at any time and potentially from many users concurrently. 

Finally, we would implement a CI/CD pipeline to build, test, and deploy new versions of the package, ensuring any updates are reliably and seemlessly deployed. Using such tools, when we are ready to update our package we can push the code to test environment where automated checks assess whether the code appears to run without crashing. These tests can include small-scale testing to ensure single functions work (unit testing) and larger-scale testing to ensure data flowing between functions and package outputs are all within reasonable bounds (integration testing). If all checks are passed, a new container for our updated package can be created and deployed to the cloud platform.

# C) Model versioning, scalability, and monitoring

Our model versioning is currently achieves a number of desirable qualities. First, our models are saved within our Optuna studies, and these Optuna studies contain important information such as datetime logs, hyperparameter logs, and custom user logs where we save the paths to our saved PyTorch models. These studies are also named with timestamps to help keep separate Optuna studies organized and help ensure there are never any accidental replacements of a study due to names. The Optuna records do not include system-/python-environment level variables however. For those I implemented MLflow to track Python environment details for each model with MLFlow artifacts. 

With respect to scalability, the cloud platform hosting our package would likely have autoscaling to adjust the number of container replicas based on traffic. However, one potential point of change to our package would depend on user behaviour. I have currently built the inference api to accept single samples but if users of the package are more interested in sending large volumes of samples through our inference API then we would experience some significant inefficiency and would need to design a batch inference API in addition to, or to replace, our single sample inference API. 

With respect to monitoring, the package currently has Loguru to capture the behaviour of the package as it runs and when paired with a monitoring tool we can track all important details related to it's behaviour. This is crucial for everything from evaluating model performance over time to debugging if problems occur while the package is deployed. 
